In [0]:
# Retrieve Azure Storage credentials securely from Databricks Secrets

storage_account_name = dbutils.secrets.get(
    scope="retail-scope",
    key="stname"
)

storage_account_key = dbutils.secrets.get(
    scope="retail-scope",
    key="stgkey"
)

print("Azure Storage credentials retrieved successfully.")

In [0]:
# ============================================================
# CELL 2: Connect to Azure Blob Storage
# ============================================================
# We use the Azure SDK directly (BlobServiceClient) instead of
# spark.conf.set(...) because this workspace runs on Serverless compute,
# which does not allow Spark to authenticate directly to Blob Storage.
# The SDK approach works around that restriction.

from azure.storage.blob import BlobServiceClient

connection_string = (
    f"DefaultEndpointsProtocol=https;"
    f"AccountName={storage_account_name};"
    f"AccountKey={storage_account_key};"
    f"EndpointSuffix=core.windows.net"
)

blob_service = BlobServiceClient.from_connection_string(connection_string)
container_client = blob_service.get_container_client("raw")  # our container is named "raw"

print("Container exists:", container_client.exists())

In [0]:
import os

sales_table = "bronze.raw_sales"
store_table = "bronze.raw_store"
quarantine_table = "bronze.quarantine_files"
ingested_files_table = "bronze.ingested_files"

current_user = spark.sql("SELECT current_user()").collect()[0][0]
workspace_dir = f"/Workspace/Users/{current_user}/capstone_tmp"
os.makedirs(workspace_dir, exist_ok=True)

print("Sales Table       :", sales_table)
print("Store Table       :", store_table)
print("Quarantine Table  :", quarantine_table)
print("Ingested Files Log:", ingested_files_table)
print("Workspace Dir     :", workspace_dir)

In [0]:
blob_list = [blob.name for blob in container_client.list_blobs()]

print(f"Files found in container: {len(blob_list)}")
for name in blob_list:
    print(" -", name)

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

sales_exists = spark.catalog.tableExists(sales_table)
store_exists = spark.catalog.tableExists(store_table)

bronze_initialized = sales_exists and store_exists

print("Sales Bronze table exists :", sales_exists)
print("Store Bronze table exists :", store_exists)
print("Bronze already initialized:", bronze_initialized)

In [0]:
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType,
    DateType, DoubleType, TimestampType
)

sales_schema = StructType([
    StructField("Store", IntegerType(), True),
    StructField("DayOfWeek", IntegerType(), True),
    StructField("Date", DateType(), True),
    StructField("Sales", IntegerType(), True),
    StructField("Customers", IntegerType(), True),
    StructField("Open", IntegerType(), True),
    StructField("Promo", IntegerType(), True),
    StructField("StateHoliday", StringType(), True),
    StructField("SchoolHoliday", IntegerType(), True),
])

store_schema = StructType([
    StructField("Store", IntegerType(), True),
    StructField("StoreType", StringType(), True),
    StructField("Assortment", StringType(), True),
    StructField("CompetitionDistance", DoubleType(), True),
    StructField("CompetitionOpenSinceMonth", IntegerType(), True),
    StructField("CompetitionOpenSinceYear", IntegerType(), True),
    StructField("Promo2", IntegerType(), True),
    StructField("Promo2SinceWeek", IntegerType(), True),
    StructField("Promo2SinceYear", IntegerType(), True),
    StructField("PromoInterval", StringType(), True),
])

ingested_files_schema = StructType([
    StructField("source_file", StringType(), True),
    StructField("ingestion_time", TimestampType(), True),
])

print("Schemas defined for sales, store, and ingested-files tracking.")

In [0]:
# ============================================================
# CELL 6: Download source files and load into Spark
# ============================================================
# Step 1: download the raw CSVs from Blob Storage into our local
#         workspace folder (required on Serverless — Spark can't read
#         directly from Blob Storage without the config we can't set).
# Step 2: read them into Spark DataFrames using our explicit schemas.

for fname in ["train.csv", "store.csv"]:
    blob_client = container_client.get_blob_client(fname)
    with open(f"{workspace_dir}/{fname}", "wb") as f:
        f.write(blob_client.download_blob().readall())

sales_df = spark.read.option("header", True).schema(sales_schema).csv(f"file:{workspace_dir}/train.csv")
store_df = spark.read.option("header", True).schema(store_schema).csv(f"file:{workspace_dir}/store.csv")

print(f"Sales rows loaded: {sales_df.count():,} (expected 1,017,209)")
print(f"Store rows loaded: {store_df.count():,} (expected 1,115)")

In [0]:
# ============================================================
# CELL 7: Add ingestion metadata and write Bronze tables
# ============================================================
# ingestion_time — when this row was loaded, useful for auditing
# source_file    — which file this row came from, useful for tracing
#                  problems back to their source
#
# mode("overwrite") replaces the table completely each time this runs,
# which is intentional for the initial load — it guarantees a clean
# table with no risk of accidental duplicates.

from pyspark.sql.functions import current_timestamp, lit

sales_bronze_df = (
    sales_df
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", lit("raw/train.csv"))
)

store_bronze_df = (
    store_df
    .withColumn("ingestion_time", current_timestamp())
    .withColumn("source_file", lit("raw/store.csv"))
)

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

sales_bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(sales_table)
store_bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(store_table)

print("Bronze tables created: bronze.raw_sales, bronze.raw_store")

In [0]:
# ============================================================
# CELL 8: Create the "ingested files" tracking table
# ============================================================
# This is our substitute for Databricks Auto Loader's checkpoint
# mechanism (Auto Loader itself needs a classic cluster, which isn't
# available here). This table remembers which files we've already
# loaded, so future runs only process genuinely new files instead
# of reloading everything and creating duplicates.

ingested_df = spark.createDataFrame(
    [("raw/train.csv", None), ("raw/store.csv", None)],
    schema=ingested_files_schema
).withColumn("ingestion_time", current_timestamp())

ingested_df.write.format("delta").mode("overwrite").saveAsTable(ingested_files_table)

print("Ingested files log created.")

In [0]:
# ============================================================
# CELL 9: Helper functions for future file processing
# ============================================================
# quarantine_file()      — logs any file we can't classify or process,
#                           instead of silently dropping it or crashing.
# align_to_bronze_schema() — makes sure a new incoming file's columns
#                           match our existing Bronze table exactly,
#                           so it can be safely appended.

from pyspark.sql import functions as F

def quarantine_file(file_path, reason):
    quarantine_df = (
        spark.createDataFrame([(file_path, reason)], ["source_file", "reason"])
        .withColumn("ingestion_time", current_timestamp())
    )
    quarantine_df.write.format("delta").mode("append").saveAsTable(quarantine_table)


def align_to_bronze_schema(df, target_table):
    target_schema = spark.table(target_table).drop("ingestion_time", "source_file").schema
    target_columns = [field.name for field in target_schema]

    for field in target_schema:
        if field.name not in df.columns:
            df = df.withColumn(field.name, F.lit(None).cast(field.dataType))
    for field in target_schema:
        df = df.withColumn(field.name, F.col(field.name).cast(field.dataType))

    return df.select(*target_columns)

print("Helper functions defined.")

In [0]:
# ============================================================
# CELL 10: Process new files (Auto Loader substitute)
# ============================================================
# Checks the container for files we haven't ingested yet (using the
# tracking table from Cell 8), classifies each by filename, aligns
# its schema, adds metadata, and appends it to the right Bronze table.
# Anything unrecognized goes to quarantine instead of breaking the run.
#
# Note: new files are read via pandas directly from downloaded bytes
# (not written to local disk first) because on Serverless compute,
# files written to /Workspace by Python aren't always immediately
# visible to Spark's separate execution backend — reading through
# pandas in memory avoids that timing issue entirely.
#
# IGNORED_FILES: these exist in the container but are deliberately
# excluded — test.csv and sample_submission.csv are Kaggle competition
# artifacts, not part of our actual data pipeline.

import pandas as pd
import io

IGNORED_FILES = {"test.csv", "sample_submission.csv"}

def process_new_files():
    already_ingested = set(
        row["source_file"] for row in spark.table(ingested_files_table).select("source_file").collect()
    )

    all_blobs = [blob.name for blob in container_client.list_blobs() if blob.name not in IGNORED_FILES]
    new_files = [f"raw/{name}" for name in all_blobs if f"raw/{name}" not in already_ingested]

    print(f"New files detected: {len(new_files)}")

    for file_path in new_files:
        file_name = file_path.split("/")[-1]
        file_name_lower = file_name.lower()

        print("\n" + "=" * 60)
        print(f"Processing file: {file_name}")

        # Classify the file by its name to decide which table it belongs to.
        if "sales" in file_name_lower or "train" in file_name_lower:
            target_table, target_schema_struct = sales_table, sales_schema
        elif "store" in file_name_lower:
            target_table, target_schema_struct = store_table, store_schema
        else:
            print(f"SKIPPED: {file_name} does not match a known pattern.")
            quarantine_file(file_path, "Unknown filename classification")
            continue

        print(f"Target table: {target_table}")

        try:
            blob_client = container_client.get_blob_client(file_name)
            blob_bytes = blob_client.download_blob().readall()

            # Read directly into pandas from memory — no local file involved
            pdf = pd.read_csv(io.BytesIO(blob_bytes))

            # Convert Date column to proper date type if present
            if "Date" in pdf.columns:
                pdf["Date"] = pd.to_datetime(pdf["Date"]).dt.date

            file_df = spark.createDataFrame(pdf, schema=target_schema_struct)
            file_df = align_to_bronze_schema(file_df, target_table)

            file_df = (file_df
                .withColumn("ingestion_time", current_timestamp())
                .withColumn("source_file", lit(file_path)))

            # append (not overwrite) — this is how new files get added
            # without wiping out what's already there.
            file_df.write.format("delta").mode("append").saveAsTable(target_table)

            # record that we've now processed this file, so it's never
            # picked up again on future runs.
            log_df = spark.createDataFrame([(file_path,)], ["source_file"]).withColumn("ingestion_time", current_timestamp())
            log_df.write.format("delta").mode("append").saveAsTable(ingested_files_table)

            print(f"Successfully appended '{file_name}' to {target_table}")

        except Exception as e:
            print(f"QUARANTINED: {file_name}")
            print(f"Reason: {str(e)}")
            quarantine_file(file_path, str(e))

        print("=" * 60)

print("process_new_files() defined.")

In [0]:
# ============================================================
# CELL 11: Run the new-files check
# ============================================================
# Since we just loaded train.csv and store.csv in Cell 6-7, and they're
# already logged in Cell 8, this should correctly report ZERO new files.
# This confirms our tracking table and container are in sync.

process_new_files()
print("Ingestion check complete.")

In [0]:
# ============================================================
# CELL 12: Validate the Bronze layer
# ============================================================
# Row counts, duplicate check, and referential integrity —
# the three checks that actually confirm the data is trustworthy,
# not just that the code ran without errors.

total_sales = spark.table(sales_table).count()
distinct_sales = spark.table(sales_table).select("Store", "Date").distinct().count()

total_store = spark.table(store_table).count()
distinct_store = spark.table(store_table).select("Store").distinct().count()

orphan_stores = (
    spark.table(sales_table).select("Store").distinct()
    .join(spark.table(store_table).select("Store"), on="Store", how="left_anti")
)

print("=== BRONZE VALIDATION ===")
print(f"Sales total rows      : {total_sales:,} (expected 1,017,209)")
print(f"Sales duplicate rows  : {total_sales - distinct_sales:,} (expected 0)")
print(f"Store total rows      : {total_store:,} (expected 1,115)")
print(f"Store duplicate rows  : {total_store - distinct_store:,} (expected 0)")
print(f"Orphan store records  : {orphan_stores.count():,} (expected 0)")

In [0]:
# Confirm distinct ingestion_time values in Bronze —
# each one represents a separate load "batch"

spark.table("bronze.raw_sales").select("ingestion_time").distinct().orderBy("ingestion_time").show(truncate=False)